In [ ]:
%run ../../imports_common.py -e "m5-forecast-ml"

In [1]:
# Wide Format: Each row = one SKU/store combo.  Each column = one day
# Long Format: Each row = one SKU on one specific day. (unique_id = SKU_STORE, timestamp = day, y = target sales)
# ADI: Average Demand Interval = average number of periods between non-zero demand (frequency demand per period)
# - determine how often does demand happen
# - Higher ADI = sparser, more intermittent demand (more zero periods between non-zero demand
# CV²: Coefficient of Variation squared = (standard deviation of demand / mean demand)²

import numpy  as np
import pandas as pd
from   pathlib import Path
from   dataclasses import asdict

from   datasetsforecast.m5 import M5
from   velari_core.core import read_root_dir
from   velari_data.datasets import DatasetTimeseries, DatasetTabular, DatasetSpecInfo, DatasetSpecTabularInfo

In [2]:
data_dir = Path(read_root_dir()) / "data"
Y_df, X_df, S_df = M5.load(directory=str(data_dir))
Y_df.head()

,unique_id,ds,y
0,FOODS_1_001_CA_1,2011-01-29,3.0
1,FOODS_1_001_CA_1,2011-01-30,0.0
2,FOODS_1_001_CA_1,2011-01-31,0.0
3,FOODS_1_001_CA_1,2011-02-01,1.0
4,FOODS_1_001_CA_1,2011-02-02,4.0


In [3]:
# Y_df is already in long format: unique_id, ds, y
# X_df: represents the exogenous variables (features) for each unique_id and ds combination
# Y_df: represents the target variable (sales) for each unique_id and ds combination
# S_df: represents the static features for each unique_id (e.g., product category, store location, etc.)
# Safety Stock is about lead-time demand

class ForecastDataset(DatasetTimeseries):
    def __init__(self, spec: DatasetSpecTabularInfo):
        super().__init__(spec=spec)

    @staticmethod
    def _load(path: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        data_dir = Path(read_root_dir()) / "data"
        Y_df, X_df, S_df = M5.load(directory=str(data_dir))
        return Y_df, X_df, S_df

    def partition(self, cutoff_date: pd.Timestamp) -> tuple[pd.DataFrame, pd.DataFrame]:
        self.df_train = self.df[self.df["ds"] <=  cutoff_date].copy()
        self.df_test  = self.df[self.df["ds"]   > cutoff_date].copy()
        return self.df_train, self.df_test

    def info(self) -> dict:
        lead_time          = 14  # expected exposure lead time in days
        forecast_horizon   = 28  # sales_train_evaluation.csv vs sales_train_validation.csv split on
        return dict(
            lead_time        = lead_time,
            forecast_horizon = forecast_horizon,
            cutoff_date      = self.df["ds"].max() - np.timedelta64(forecast_horizon, 'D'),
            num_series       = self.df["unique_id"].nunique(),
            span = asdict(self.temporal_span(column="ds", unit="D")),
        )


spec = DatasetSpecTabularInfo(
    info=DatasetSpecInfo(
        data=Y_df,
        name="m5-forecast-ml",
        description="M5 Forecasting Competition dataset in long format for ML modeling",
    )
)
dataset = ForecastDataset(spec=spec)
info    = dataset.info()
part    = dataset.partition(cutoff_date=info["cutoff_date"])
